In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score

In [2]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [3]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [4]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [5]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 2.5  # This gives 4.0 seconds.

In [6]:
train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
train_active_y = []         # List to hold event labels per subject.
train_active_metadata = []  # List to hold event metadata per subject.

# Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
for subj in range(1, 10):
    filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
    
    # Read the GDF file (using preload=True to load data into memory).
    train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
    # Retain only EEG channels (22 channels) and exclude EOG channels.
    train_raw.pick_types(eeg=True, eog=False)
    
    # Extract events corresponding only to the four desired types.
    train_active_events, _ = mne.events_from_annotations(train_raw, event_id=active_lr_event_ids)
    
    # Create epochs from tmin to tmax.
    train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=active_lr_event_ids, tmin=tmin, tmax=tmax,
                        baseline=None, preload=True, verbose=False)
    
    # Get the epoch data (num_epochs x 22 channels x 1001 samples).
    train_active_data = train_active_epochs.get_data()
    
    # Print the number of extracted epochs to verify
    print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
    
    # Sampling frequency from raw.info (should be 250).
    fs = int(train_raw.info['sfreq'])
    # print(fs)
    n_trials, n_channels, n_times = train_active_data.shape
    train_active_filtered_data = np.empty_like(train_active_data)
    
    # Apply the causal bandpass filter channel‐wise for each trial.
    for trial in range(n_trials):
        for ch in range(n_channels):
            train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                train_active_data[trial, ch, :],
                lowcut=8,    # Lower bound of sensorimotor rhythm.
                highcut=30,  # Upper bound of sensorimotor rhythm.
                fs=fs,
                order=5     # Lower order for a smoother causal filter.
            )
    # Apply the causal bandpass filter channel‐wise for each trial.
    # for trial in range(n_trials):
    #     for ch in range(n_channels):
    #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
    
    # Append the processed data, labels, and event metadata.
    train_active_X.append(train_active_filtered_data)
    train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
    train_active_metadata.append(train_active_epochs.events)

print("Loaded data for", len(train_active_X), "subjects.")

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 7: Epoch data shape (144, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 8: Epoch data shape (144, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 9: Epoch data shape (144, 22, 501)
Loaded data for 9 subjects.


In [7]:
eval_active_X = []         
eval_active_y = []        
eval_active_metadata = [] 

# Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
for subj in range(1, 10):
    filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
    mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
    true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
    # Read the GDF file (using preload=True to load data into memory).
    eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
    # Retain only EEG channels (22 channels) and exclude EOG channels.
    eval_raw.pick_types(eeg=True, eog=False)
    
    # Extract events corresponding only to the four desired types.
    eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
    
    # Create epochs from tmin to tmax.
    eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                        baseline=None, preload=True, verbose=False)
    
    # Get the epoch data (num_epochs x 22 channels x 1001 samples).
    eval_active_data = eval_active_epochs.get_data()
    
    # Print the number of extracted epochs to verify
    print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
    
    # Sampling frequency from raw.info (should be 250).
    fs = int(eval_raw.info['sfreq'])
    # print(fs)
    n_trials, n_channels, n_times = eval_active_data.shape
    eval_active_filtered_data = np.empty_like(eval_active_data)
    
    # Apply the causal bandpass filter channel‐wise for each trial.
    for trial in range(n_trials):
        for ch in range(n_channels):
            eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                eval_active_data[trial, ch, :],
                lowcut=8,    # Lower bound of sensorimotor rhythm.
                highcut=30,  # Upper bound of sensorimotor rhythm.
                fs=fs,
                order=5     # Lower order for a smoother causal filter.
            )
    
    # for trial in range(n_trials):
    #     for ch in range(n_channels):
    #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                
    
    # Append the processed data, labels, and event metadata.
    eval_active_X.append(eval_active_filtered_data)
    eval_active_y.append(true_y)  # The third column holds the event code.
    eval_active_metadata.append(eval_active_epochs.events)

print("Loaded data for", len(eval_active_X), "subjects.")

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 1: Epoch data shape (288, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 2: Epoch data shape (288, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 501)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 501)
Loaded data for 9 subjects.


In [ ]:
# eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
# eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]

In [8]:
# Before running TSA
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

# Encode both train and eval labels
train_active_y = encode_labels(train_active_y)
eval_active_y = encode_labels(eval_active_y)

# Then use these encoded labels with the TSA function

In [18]:
from pyriemann.utils import mean_logeuclid
from pyriemann.utils.tangentspace import tangent_space

In [24]:
import numpy as np
from pyriemann.estimation import Covariances
from pyriemann.utils.mean import mean_logeuclid
from pyriemann.utils.tangentspace import tangent_space
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Initialize variables
subject_accuracies = []
align_per_class = 14
n_subjects = 9
classes = [0, 1]  # Four classes as indicated by the loop over classes

# Loop over each subject
for subj in range(n_subjects):
    # Load source and target data
    X_source = train_active_X[subj]  # Shape: (288, 22, 1001)
    y_source = train_active_y[subj]  # Shape: (288,)
    X_target = eval_active_X[subj]  # Shape: (288, 22, 1001)
    y_target = eval_active_y[subj]  # Shape: (288,)

    # Split target data into alignment and test sets
    align_indices = []
    test_indices = []
    for cls in classes:
        cls_indices = np.where(y_target == cls)[0]
        np.random.shuffle(cls_indices)
        align_indices.extend(cls_indices[:align_per_class])
        test_indices.extend(cls_indices[align_per_class:])
    
    X_target_align = X_target[align_indices]  # Shape: (56, 22, 1001)
    y_target_align = y_target[align_indices]  # Shape: (56,)
    X_target_test = X_target[test_indices]    # Shape: (232, 22, 1001)
    y_target_test = y_target[test_indices]    # Shape: (232,)

    # Compute covariance matrices using pyriemann
    cov_estimator = Covariances(estimator='scm')  # Sample Covariance Matrix estimator
    C_source = cov_estimator.fit_transform(X_source)          # Shape: (288, 22, 22)
    C_target_align = cov_estimator.fit_transform(X_target_align)  # Shape: (56, 22, 22)
    C_target_test = cov_estimator.fit_transform(X_target_test)    # Shape: (232, 22, 22)

    # Compute Log-Euclidean means using pyriemann
    M_source = mean_logeuclid(C_source)        # Shape: (22, 22)
    M_target = mean_logeuclid(C_target_align)  # Shape: (22, 22)

    # Project to tangent space using pyriemann with affine-invariant metric
    S_source = tangent_space(C_source, M_source, metric='riemann')          # Shape: (288, 253)
    S_target_align = tangent_space(C_target_align, M_target, metric='riemann')  # Shape: (56, 253)
    S_target_test = tangent_space(C_target_test, M_target, metric='riemann')    # Shape: (232, 253)

    # Rescale tangent space vectors to average norm of 1
    average_norm_source = np.mean(np.linalg.norm(S_source, axis=1))
    bar_s_source = S_source / average_norm_source  # Shape: (288, 253)

    average_norm_target = np.mean(np.linalg.norm(S_target_align, axis=1))
    bar_t_target_align = S_target_align / average_norm_target  # Shape: (56, 253)
    bar_t_target_test = S_target_test / average_norm_target    # Shape: (232, 253)

    # Compute class means for alignment
    bar_s_k = [np.mean(bar_s_source[y_source == k], axis=0) for k in classes]  # List of 4 vectors, each (253,)
    bar_t_k = [np.mean(bar_t_target_align[y_target_align == k], axis=0) for k in classes]  # List of 4 vectors, each (253,)

    # Stack class means into matrices
    bar_S = np.column_stack(bar_s_k)  # Shape: (253, 4)
    bar_T = np.column_stack(bar_t_k)  # Shape: (253, 4)

    # Compute cross-product matrix
    C_st = bar_S @ bar_T.T  # Shape: (253, 253)

    # Perform SVD on cross-product matrix
    U, D, Vh = np.linalg.svd(C_st, full_matrices=False)

    # Select top singular vectors explaining 99.9% of variance
    cumsum_D = np.cumsum(D)
    total_D = np.sum(D)
    N_v = np.searchsorted(cumsum_D, 0.999 * total_D) + 1
    if N_v == 0:
        N_v = 1  # Ensure at least one singular vector

    # Construct rotation matrix
    rotation_matrix = U[:, :N_v] @ Vh[:N_v, :]  # Shape: (253, 253)

    # Align target test vectors
    hat_t_test = bar_t_target_test @ rotation_matrix.T  # Shape: (232, 253)

    # Train SVC on source data and predict on aligned test data
    clf = SVC(kernel='linear')
    clf.fit(bar_s_source, y_source)
    y_pred = clf.predict(hat_t_test)

    # Compute accuracy
    acc = accuracy_score(y_target_test, y_pred)
    subject_accuracies.append(acc)

    # Optional: Compute accuracy without alignment for comparison
    y_pred_no_align = clf.predict(bar_t_target_test)
    print(f"Subject {subj} - No-alignment accuracy: {accuracy_score(y_target_test, y_pred_no_align):.4f}")
    print(f"Subject {subj} - Aligned accuracy: {acc:.4f}")

# Compute and print average accuracy across subjects
mean_accuracy = np.mean(subject_accuracies)
print(f"Average accuracy across {n_subjects} subjects: {mean_accuracy:.4f}")

Subject 0 - No-alignment accuracy: 0.7672
Subject 0 - Aligned accuracy: 0.7586
Subject 1 - No-alignment accuracy: 0.6293
Subject 1 - Aligned accuracy: 0.5862
Subject 2 - No-alignment accuracy: 0.9052
Subject 2 - Aligned accuracy: 0.8793
Subject 3 - No-alignment accuracy: 0.6810
Subject 3 - Aligned accuracy: 0.4741
Subject 4 - No-alignment accuracy: 0.5776
Subject 4 - Aligned accuracy: 0.5086
Subject 5 - No-alignment accuracy: 0.6293
Subject 5 - Aligned accuracy: 0.5948
Subject 6 - No-alignment accuracy: 0.5862
Subject 6 - Aligned accuracy: 0.5862
Subject 7 - No-alignment accuracy: 0.9655
Subject 7 - Aligned accuracy: 0.9569
Subject 8 - No-alignment accuracy: 0.9138
Subject 8 - Aligned accuracy: 0.8966
Average accuracy across 9 subjects: 0.6935


In [14]:
for i in range(n_subjects):
    print(f"Subject {i+ 1}: Accuracy = {subject_accuracies[i]:.4f}")
print("mean_accuracy: ", np.mean(subject_accuracies))

Subject 1: Accuracy = 0.5000
Subject 2: Accuracy = 0.5172
Subject 3: Accuracy = 0.5345
Subject 4: Accuracy = 0.5000
Subject 5: Accuracy = 0.5000
Subject 6: Accuracy = 0.5000
Subject 7: Accuracy = 0.4914
Subject 8: Accuracy = 0.4914
Subject 9: Accuracy = 0.5000
mean_accuracy:  0.5038314176245211
